# 05 - Embeddings con PCA

Aplica PCA con el umbral determinado empíricamente en el notebook 04
y genera los embeddings definitivos para el entrenamiento de modelos.

**Entrada**: `data/embeddings/train_embeddings_raw.parquet`, `test_embeddings_raw.parquet`  
**Salida**: `data/embeddings/train_embeddings.parquet`, `test_embeddings.parquet`, `models/pca_model.joblib`

---

**CONFIGURACIÓN**: Ajustá `USE_SUBSET` y `PCA_VARIANCE` según el notebook 04.
- `USE_SUBSET` debe coincidir con el notebook 03.
- `PCA_VARIANCE` viene de la decisión documentada en `decisions/pca_variance.md`.

## 1. Importaciones

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import joblib
from pathlib import Path

print('✓ Librerías importadas')

✓ Librerías importadas


## 2. Configuración

In [2]:
# ============================================================
# CONFIGURACIÓN — debe coincidir con notebook 03
# ============================================================
USE_SUBSET   = False  # True = subset | False = dataset completo
PCA_VARIANCE = 0.95   # Umbral elegido en notebook 04
# ============================================================

DATA_EMBEDDINGS = Path('../data/embeddings')
MODELS_DIR      = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

suffix = '_subset' if USE_SUBSET else ''

print('=' * 55)
print('CONFIGURACIÓN')
print('=' * 55)
print(f'  Modo         : {"SUBSET" if USE_SUBSET else "COMPLETO"}')
print(f'  PCA varianza : {PCA_VARIANCE:.0%}')
print(f'  Entrada      : train_embeddings_raw{suffix}.parquet')
print(f'  Salida       : train_embeddings{suffix}.parquet')
print('=' * 55)

CONFIGURACIÓN
  Modo         : COMPLETO
  PCA varianza : 95%
  Entrada      : train_embeddings_raw.parquet
  Salida       : train_embeddings.parquet


## 3. Carga de Embeddings Crudos

In [3]:
print('Cargando embeddings crudos...')
train_df = pd.read_parquet(DATA_EMBEDDINGS / f'train_embeddings_raw{suffix}.parquet')
test_df  = pd.read_parquet(DATA_EMBEDDINGS / f'test_embeddings_raw{suffix}.parquet')

emb_cols = [c for c in train_df.columns if c.startswith('emb_')]
X_train  = train_df[emb_cols].values
X_test   = test_df[emb_cols].values
y_train  = train_df['emotion'].values
y_test   = test_df['emotion'].values

print(f'  Train : {X_train.shape}')
print(f'  Test  : {X_test.shape}')

Cargando embeddings crudos...
  Train : (441127, 384)
  Test  : (110282, 384)


## 4. Aplicar PCA

In [4]:
print(f'Entrenando PCA (varianza={PCA_VARIANCE:.0%})...')
pca = PCA(n_components=PCA_VARIANCE, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

var_real = pca.explained_variance_ratio_.sum()
n_dims   = X_train_pca.shape[1]

print(f'  Dims originales  : {X_train.shape[1]}')
print(f'  Dims resultantes : {n_dims}')
print(f'  Varianza real    : {var_real:.4f}')
print(f'  Reducción        : {X_train.shape[1] - n_dims} dims ({(X_train.shape[1] - n_dims)/X_train.shape[1]*100:.0f}%)')

Entrenando PCA (varianza=95%)...
  Dims originales  : 384
  Dims resultantes : 232
  Varianza real    : 0.9504
  Reducción        : 152 dims (40%)


## 5. Carga y Estandarización de Features Numéricos

Los embeddings capturan el contenido semántico de las letras, pero el dataset
también contiene features acústicos (Tempo, Energy, Loudness, etc.) y atributos
contextuales (Good for Party, Good for Exercise, etc.) que aportan información
complementaria sobre la emoción de una canción.

**Decisión técnica**: aplicar `StandardScaler` sobre estos features numéricos
(media=0, desviación=1) antes de concatenarlos con los embeddings PCA.

**Justificación**:
- Los embeddings PCA ya están centrados (PCA centra automáticamente).
- Los features numéricos tienen escalas muy distintas: Tempo (60–200),
  Loudness (−20 a 0), Genre_freq (0–1). Sin estandarización, el modelo
  le daría más peso a las variables de mayor magnitud.
- El scaler se fitea **solo sobre train** para evitar data leakage.
- Las columnas de texto (`text`, `text_clean`) se excluyen — ya fueron
  procesadas en el notebook 03 para generar los embeddings.

In [5]:
from sklearn.preprocessing import StandardScaler

DATA_PROCESSED = Path('../data/processed')

# Columnas a excluir (texto y target)
EXCLUDE_COLS = ['text', 'text_clean', 'emotion']

# Cargar datos procesados
print('Cargando features numéricos desde processed...')
train_proc = pd.read_parquet(DATA_PROCESSED / 'train.parquet')
test_proc  = pd.read_parquet(DATA_PROCESSED / 'test.parquet')

# Columnas numéricas (todo excepto texto y target)
numeric_cols = [c for c in train_proc.columns if c not in EXCLUDE_COLS]

print(f'  Features numéricos : {len(numeric_cols)} columnas')
print(f'  Train              : {train_proc[numeric_cols].shape}')
print(f'  Test               : {test_proc[numeric_cols].shape}')
print()

# Estandarizar — fit SOLO sobre train, transform sobre ambos
scaler = StandardScaler()
X_train_num = scaler.fit_transform(train_proc[numeric_cols].values)
X_test_num  = scaler.transform(test_proc[numeric_cols].values)

print('Estandarización completada:')
print(f'  Media train (primeras 3 cols): {X_train_num[:, :3].mean(axis=0).round(4)}')
print(f'  Std  train (primeras 3 cols):  {X_train_num[:, :3].std(axis=0).round(4)}')
print()

# Guardar scaler para reproducibilidad y uso en inferencia
scaler_path = MODELS_DIR / f'scaler_model{suffix}.joblib'
joblib.dump(scaler, scaler_path)
print(f'  ✓ {scaler_path.name} guardado')
print()
print('Columnas estandarizadas:')
for col in numeric_cols:
    print(f'  - {col}')


Cargando features numéricos desde processed...
  Features numéricos : 25 columnas
  Train              : (441127, 25)
  Test               : (110282, 25)

Estandarización completada:
  Media train (primeras 3 cols): [ 0.  0. -0.]
  Std  train (primeras 3 cols):  [1. 1. 1.]

  ✓ scaler_model.joblib guardado

Columnas estandarizadas:
  - Tempo
  - Popularity
  - Energy
  - Danceability
  - Positiveness
  - Speechiness
  - Liveness
  - Acousticness
  - Instrumentalness
  - Good for Party
  - Good for Work/Study
  - Good for Relaxation/Meditation
  - Good for Exercise
  - Good for Running
  - Good for Yoga/Stretching
  - Good for Driving
  - Good for Social Gatherings
  - Good for Morning Routine
  - Length_seconds
  - Loudness
  - Time_signature
  - Key_tonic
  - Key_mode
  - Explicit_binary
  - Genre_freq


## 6. Concatenación: Embeddings PCA + Features Numéricos

Se combinan horizontalmente los embeddings reducidos por PCA con los features
numéricos estandarizados. El resultado es la matriz de features definitiva
que usarán los modelos de clasificación (notebooks 06 y 07).

**Estructura final**:
- Primeras `N` columnas: embeddings semánticos de las letras (reducidos por PCA)
- Últimas `M` columnas: features acústicos y contextuales (estandarizados con StandardScaler)

In [6]:
# Concatenar embeddings PCA con features numéricos estandarizados
X_train_combined = np.hstack([X_train_pca, X_train_num])
X_test_combined  = np.hstack([X_test_pca,  X_test_num])

n_emb = X_train_pca.shape[1]
n_num = X_train_num.shape[1]

print('=' * 55)
print('MATRIZ DE FEATURES FINAL')
print('=' * 55)
print(f'  Embeddings PCA   : {n_emb} dims')
print(f'  Features numéric.: {n_num} dims')
print(f'  TOTAL            : {X_train_combined.shape[1]} dims')
print()
print(f'  Train shape      : {X_train_combined.shape}')
print(f'  Test  shape      : {X_test_combined.shape}')
print('=' * 55)


MATRIZ DE FEATURES FINAL
  Embeddings PCA   : 232 dims
  Features numéric.: 25 dims
  TOTAL            : 257 dims

  Train shape      : (441127, 257)
  Test  shape      : (110282, 257)


## 7. Guardar Embeddings y Modelo PCA

In [7]:
def save_embeddings(X, y, path):
    df = pd.DataFrame(X, columns=[f'emb_{i}' for i in range(X.shape[1])])
    df['emotion'] = y
    df.to_parquet(path, index=False)
    mb = path.stat().st_size / 1e6
    print(f'  ✓ {path.name} — {df.shape} — {mb:.0f} MB')

print('Guardando embeddings reducidos...')
save_embeddings(X_train_combined, y_train, DATA_EMBEDDINGS / f'train_embeddings{suffix}.parquet')
save_embeddings(X_test_combined,  y_test,  DATA_EMBEDDINGS / f'test_embeddings{suffix}.parquet')

pca_path = MODELS_DIR / f'pca_model{suffix}.joblib'
joblib.dump(pca, pca_path)
print(f'  ✓ {pca_path.name}')

print()
print(f'✓ Listo. Features combinados de {X_train_combined.shape[1]} dims '
      f'({n_emb} emb + {n_num} numéricos) listos para notebooks 06 y 07.')

Guardando embeddings reducidos...
  ✓ train_embeddings.parquet — (441127, 258) — 693 MB
  ✓ test_embeddings.parquet — (110282, 258) — 203 MB
  ✓ pca_model.joblib

✓ Listo. Features combinados de 257 dims (232 emb + 25 numéricos) listos para notebooks 06 y 07.
